# VALORANT 前日試合ふり返り
ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んでから、上から2つ実行するだけです。
ランタイムを再起動・切り替えたときも、必ずセル1 → セル2 の順に実行してください（セル1の最初で Drive を接続します。認証の画面で許可したら、あとは最後まで放っておけます）。
設定は `config/default.yaml`、上書きは Drive の `valorant-replay/config.yaml` に書きます。

In [ ]:
# ① 最初に Drive を接続（認証の画面が出るのはここだけ。許可したら後は放っておいて大丈夫）
from google.colab import drive, userdata
import os, shutil
if not os.path.ismount('/content/drive'):
    if os.path.exists('/content/drive'):
        shutil.move('/content/drive', '/content/drive_local_backup')  # 未マウント時に Colab 内へできたフォルダを退避
    drive.mount('/content/drive')
def secret(k, required=False):
    try:
        v = (userdata.get(k) or '').strip()
    except Exception as e:   # 未登録・ノートブックからのアクセスがオフ・名前違い など
        if required:
            print(f'⚠ シークレット {k} を読めません: {type(e).__name__} {e}')
        return ''
    if required:
        print(f'✓ {k} を読み込みました（{len(v)} 文字, 先頭 {v[:4]}…）')
    return v
os.environ['NOTION_TOKEN'] = secret('NOTION_TOKEN', required=True)
os.environ['YT_COOKIES'] = secret('YT_COOKIES')  # 任意。Drive の valorant-replay/cookies.txt でもよい
# 任意。プレイブックの ATK/DEF summary を書かせる無料の LLM（どちらか 1 つで OK）
os.environ['OPENROUTER_API_KEY'] = secret('OPENROUTER_API_KEY')  # https://openrouter.ai/keys （そのとき無料のモデルを自動で選ぶ）
os.environ['GEMINI_API_KEY'] = secret('GEMINI_API_KEY')          # https://aistudio.google.com/apikey （Gemini 無料枠）
os.environ['VREPLAY_CONFIG'] = '/content/drive/MyDrive/valorant-replay/config.yaml'  # 無ければ既定値で動く

# ② コードの取得とインストール
!git -C /content/valorant-replay pull -q 2>/dev/null || git clone -q https://github.com/AK47bbddk/valorant-replay /content/valorant-replay
!pip -q install -e '/content/valorant-replay[video]'
!test -x ~/.deno/bin/deno || (curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null 2>&1)  # YouTube の解読に使う JS ランタイム
!bash /content/valorant-replay/scripts/pot_server.sh  # YouTube の PO トークン（無いと 360p しか取れない）
!nvidia-smi -L >/dev/null 2>&1 && ! python -c "import onnxruntime as o; assert 'CUDAExecutionProvider' in o.get_available_providers()" 2>/dev/null && (pip -q uninstall -y onnxruntime; pip -q install --force-reinstall --no-deps onnxruntime-gpu || pip -q install --force-reinstall onnxruntime)  # GPU があれば OCR も GPU で（失敗したら CPU 版に戻す）
%cd /content/valorant-replay

In [ ]:
# 前日の試合を解析（Drive とトークンはセル1で設定済み）
!vreplay run   # 例: --date 2026-10-21 / --match <VLRのURL> / --no-video / --force

### プレイブック（チーム × マップ × 構成ごとのスライド）
毎日の実行（セル2）で解析結果がウェアハウス（Drive の `valorant-replay/warehouse.sqlite3`）にたまります。
下のセルの `TEAM`・`MAP` を変えて実行すると、Strategy（strong / weak point）・配置・デフォルト・セット・リテイク・ポストプラントのスライドを作って表示し、HTML をダウンロードします（印刷すると 1 スライド 1 ページの PDF）。
`--sync` は以前の実行の report.json もまとめて取り込みます（初回だけで OK）。
`NOTION_PARENT` に親ページの URL を入れると、Notion にも ATK / DEF のデータベース（型ごとのページ・例のラウンドの動画・位置の画像）つきで書きます。シークレットに `OPENROUTER_API_KEY` か `GEMINI_API_KEY`（どちらも無料）があれば、ATK / DEF summary の文章を LLM が数字から書きます。
冊子の最初に「マップのメタ（エリアの取り方・チョークの守り方）」「じゃんけん（攻めの手 × 守りの手の相性）」「この構成の特徴（強み・弱み・アンチピック）」、後ろに「手のディテール」「ターンの応酬」「読み替え」が入ります。手の名前と相性は `config/kb/maps/<map>.yaml`（Drive の `valorant-replay/kb/maps/` に同じ形で置けば上書き）に書きます。

In [ ]:
TEAM, MAP = 'G2', 'Split'   # 空文字 '' にすると全部
NOTION_PARENT = ''          # Notion にも書くなら、プレイブックを置く親ページの URL（空なら HTML だけ）
ARGS = (f'--team "{TEAM}" ' if TEAM else '') + (f'--map "{MAP}" ' if MAP else '') + (f'--notion "{NOTION_PARENT}"' if NOTION_PARENT else '')
!vreplay kb --pull  # Obsidian のメモ（Valo-wiki）を取ってくる：エリアの評価・セオリー・Tier
!vreplay playbook --sync $ARGS
import glob, os
from IPython.display import HTML, display
from google.colab import files
d = '/content/drive/MyDrive/valorant-replay/playbook/'
decks = sorted(glob.glob(d + 'team_*.html'), key=os.path.getmtime, reverse=True)
for f in decks[:3]:
    print(f)
    display(HTML(open(f, encoding='utf-8').read()))
    files.download(f)

### 試しに 1 チーム × 1 マップを解析して、v1 と v2 のプレイブックを比べる
`MATCHES` の試合のうち `MAP` だけを解析し（`vreplay run --map`）、同じ DB から v1（v2 の直前のコミット）と v2 のプレイブックを作って並べます。v1 は `playbook_v1/`、v2 は `playbook/` に出ます。

In [ ]:
TEAM, MAP = 'LOUD', 'Sunset'
MATCHES = ['https://www.vlr.gg/754737/loud-vs-team-vitality-valorant-champions-2026-gf',
           'https://www.vlr.gg/754735/nrg-vs-loud-valorant-champions-2026-ubsf',
           'https://www.vlr.gg/754733/paper-rex-vs-loud-valorant-champions-2026-ubqf']
M = ' '.join(MATCHES)
!vreplay run --match $M --map $MAP
!vreplay kb --pull
!vreplay playbook --team $TEAM --map $MAP --no-llm
!vreplay meta --map $MAP
# v1（v2 の直前）で同じ DB からプレイブックを作る
!test -d /content/vr_v1 || git -C /content/valorant-replay worktree add -q /content/vr_v1 3832f22
!cd /content/vr_v1 && python -m vreplay playbook --team $TEAM --map $MAP --no-llm --out /content/drive/MyDrive/valorant-replay/playbook_v1
import glob, os
from IPython.display import HTML, display
for d in ('playbook_v1', 'playbook'):
    fs = sorted(glob.glob(f'/content/drive/MyDrive/valorant-replay/{d}/team_*{MAP}*.html'), key=os.path.getmtime, reverse=True)
    if fs:
        print(d, fs[0])
        display(HTML(open(fs[0], encoding='utf-8').read()))


### マップのメタ（全チームの手・相性・読み替え）
チームを問わず、そのマップで使われている攻め・守りの手と、手 × 手の勝率（じゃんけん）、負けた次のラウンドに何を出したかをまとめます。知識ベースの見立て（▲▼）と数字が合っているかもここで確かめます。

In [ ]:
MAP = 'Sunset'   # 空文字 '' にすると全マップ
!vreplay meta {('--map ' + MAP) if MAP else ''}
import glob
from IPython.display import HTML, display
for f in sorted(glob.glob('/content/drive/MyDrive/valorant-replay/playbook/meta_*.html'))[:2]:
    print(f)
    display(HTML(open(f, encoding='utf-8').read()))


### スクリムの動画を解析する（VLR なし）
Drive に置いた動画（観戦視点の録画がいちばん読める）か YouTube の URL を渡すと、ラウンドに分けて同じ解析をし、DB に入れます（source=scrim）。画面の左がチーム1（逆なら `--team1-right`）。`--start-side` はチーム1 の前半の側。

In [ ]:
VIDEO = '/content/drive/MyDrive/scrim/2026-11-05_sunset.mp4'
!vreplay scrim --video "$VIDEO" --map Sunset --team1 OURS --team2 OPP --start-side 攻撃 \
  --agents1 Chamber,Sage,Viper,Sova,Neon --agents2 Jett,Omen,Fade,Killjoy,Phoenix


### 条件に合うラウンドを抜き出す（プレイブック用）
例：自分たちの攻めで「B スプリット」を出して勝ったラウンド。`--source scrim --clips` でスクリムの動画からそのラウンドを切り出します。結果は Drive の `valorant-replay/picks/`（HTML と CSV）。

In [ ]:
!vreplay pick --map Sunset --team OURS --side 攻撃 --hand "B スプリット" --win


### ラウンドDB（Notion で全ラウンドを検索）
解析済みの全マップの各ラウンドを、Notion のデータベース（1 行＝1 ラウンド×1 チーム）に入れます。初回だけ `ROUNDS_PARENT` に置き場所の親ページの URL を入れてください（2 回目以降は空で OK）。
Notion のフィルターで「ULTラウンド にチェック」「種類＝アンチエコ」「分岐＝フェイク」「対オペ」「チーム＝G2」などを組み合わせて検索できます。毎日の実行（セル2）でも自動で追加されます。

In [ ]:
ROUNDS_PARENT = ''   # 例: https://www.notion.so/xxxx（初回だけ）。期間を絞るなら ARGS に --from 2026-09-24 --to 2026-10-19
ARGS = (f'--notion "{ROUNDS_PARENT}"' if ROUNDS_PARENT else '')
!vreplay rounds $ARGS


### ROI の確認（HUD の位置がずれているとき）
下のセルの URL を好きな試合に変えて実行すると、枠を描いた画像がその場に表示され、ダウンロードもされます。

※ ランタイムを切り替える・切断する前に最後のセルを実行してください（Drive への書き込みが反映されずに消えることがあります）。

In [ ]:
!vreplay calibrate --match https://www.vlr.gg/754737/loud-vs-team-vitality-valorant-champions-2026-gf --map 1 --at 600
import os
from IPython.display import Image, display
from google.colab import files
d = '/content/drive/MyDrive/valorant-replay/calibrate/'
for n in ('rois.jpg', 'panels.jpg', 'table.jpg'):
    if os.path.exists(d + n):
        display(Image(d + n))
        files.download(d + n)

In [ ]:
drive.flush_and_unmount()  # Drive への書き込みを確定（終わる前・ランタイムを変える前に）